# Train FarmNet (DeepLabV3) for EUDR Compliance

This notebook trains the segmentation model to identify forests, crops, and shrubbery using Sentinel-2 imagery and GEE Hybrid Masks.

## 🚀 Getting Started (Google Colab)

To use this notebook in Colab with **no code changes**:
1.  **Upload Project**: Drag and drop your entire `EUDR-compliance` folder into the root of your **Google Drive** (`My Drive`).
2.  **Open Notebook**: Open `notebooks/train_farm_net.ipynb` in Colab.
3.  **Run All**: Click 'Runtime' > 'Run all'.

The notebook will automatically mount your Drive, find the data, install dependencies, and start training.

In [ ]:
import os
import sys

# --- COLAB SETUP ---
try:
    from google.colab import drive
    IN_COLAB = True
    print("🔥 Running in Google Colab")
    
    # Mount Drive
    drive.mount('/content/drive')
    
    # Set Project Path (Adjust 'EUDR-compliance' if your folder name is different)
    # Assuming you uploaded the project folder to the root of MyDrive
    PROJECT_PATH = '/content/drive/MyDrive/EUDR-compliance'
    
    if not os.path.exists(PROJECT_PATH):
        print(f"⚠️ Warning: Project path {PROJECT_PATH} not found. Please upload your project to Drive or adjust the path.")
    else:
        os.chdir(PROJECT_PATH)
        print(f"✅ Changed directory to {PROJECT_PATH}")
        
    # Install requirements if in Colab
    !pip install rasterio torch torchvision numpy matplotlib
    
except ImportError:
    IN_COLAB = False
    print("💻 Running locally")
    PROJECT_PATH = os.path.abspath('..')

# Add project root to path to import local modules
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

# Also add GEE_dynamic for preprocessing
if os.path.join(PROJECT_PATH, 'GEE_dynamic') not in sys.path:
    sys.path.append(os.path.join(PROJECT_PATH, 'GEE_dynamic'))

import torch
import matplotlib.pyplot as plt
import numpy as np

from src.ML_farm_net import train_model, get_deeplab_model
from GEE_dynamic.preprocessing.dataset_loader import FarmSegmentationDataset

## Configuration

In [ ]:
# Paths
# If in Colab, paths are relative to PROJECT_PATH (which we chdir'd into)
# If local, PROJECT_PATH is '..'
if IN_COLAB:
    RAW_DIR = os.path.join(PROJECT_PATH, 'data', 'raw_satellite', '2020_baseline')
    MASK_DIR = os.path.join(PROJECT_PATH, 'data', 'hybrid_masks')
    MODEL_MSG_PATH = os.path.join(PROJECT_PATH, 'models', 'farm_deeplab_notebook.pth')
else:
    RAW_DIR = os.path.join(PROJECT_PATH, 'data', 'raw_satellite', '2020_baseline')
    MASK_DIR = os.path.join(PROJECT_PATH, 'data', 'hybrid_masks')
    MODEL_MSG_PATH = os.path.join(PROJECT_PATH, 'models', 'farm_deeplab_notebook.pth')

# Hyperparameters
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4

print(f"Training on data from: {RAW_DIR}")

In [ ]:
# Verify Dataset Loading
dataset = FarmSegmentationDataset(RAW_DIR, MASK_DIR, cache_aligned_masks=True)
print(f"Found {len(dataset)} valid image-mask pairs.")

if len(dataset) > 0:
    img, mask = dataset[0]
    print(f"Image shape: {img.shape}, Mask shape: {mask.shape}")
    
    # Visualize first sample
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    # Show RGB (Bands 0, 1, 2) - Normalized usually, maybe brighten
    rgb = img[:3].permute(1, 2, 0).numpy()
    # Clip to 0-0.3 for display (Sentinel 2 is dark)
    rgb = np.clip(rgb / 3000.0, 0, 1) if rgb.max() > 1 else rgb # Heuristic normalization for display
    plt.imshow(rgb)
    plt.title("Sentinel-2 RGB")
    
    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='tab10', vmin=0, vmax=10)
    plt.title("Hybrid Mask (inc. Clouds)")
    plt.colorbar()
    plt.show()

## Training Loop

In [ ]:
# Run Training
train_model(
    raw_dir=RAW_DIR,
    mask_dir=MASK_DIR,
    output_model_path=MODEL_MSG_PATH,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR
)